In [2]:
# Import necessary libraries
import requests
import pandas as pd
from IPython.display import display, HTML

# Define the SPARQL query to get 1000 chemical compounds
# This query retrieves compounds with their English labels, chemical formulas, CAS numbers, and SMILES notations

query = """
SELECT DISTINCT ?compound ?compoundLabel WHERE {
  # Find items that are chemical compounds
  {
    ?compound wdt:P31 wd:Q113145171.
  }
  UNION
  {
    ?compound wdt:P31 wd:Q11173.
  }
  UNION
  {
    ?compound wdt:P31 wd:Q11344.
  }
  
  # Optional properties to retrieve
#   OPTIONAL { ?compound wdt:P274 ?formula. }     # Chemical formula
#   OPTIONAL { ?compound wdt:P231 ?cas. }         # CAS Registry Number
#   OPTIONAL { ?compound wdt:P233 ?smiles. }      # SMILES notation
#   OPTIONAL { ?compound wdt:P234 ?inchi. }       # InChI notation
  
  # Get labels in English
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
  
  # Filter out compounds without labels
#  FILTER(BOUND(?compoundLabel))
}
LIMIT 150000
"""

# Set up the endpoint and request parameters
url = 'https://query.wikidata.org/sparql'
headers = {
    'User-Agent': 'Chemical Compounds Data Collection/1.0 (https://github.com/your-username; your-email@example.com)'
}
params = {
    'query': query,
    'format': 'json'
}

# Execute the query and get results
print("Querying Wikidata...")
response = requests.get(url, headers=headers, params=params)

# Check if request was successful
if response.status_code == 200:
    data = response.json()
    
    # Process the results into a pandas DataFrame
    results = data['results']['bindings']
    compounds = []
    
    for result in results:
        compound_info = {
            'compound_id': result['compound']['value'].split('/')[-1],
            'name': result.get('compoundLabel', {}).get('value', 'N/A'),
        }
        if compound_info['name'] != compound_info['compound_id']:
            compounds.append(compound_info['name'])
            
    # Create DataFrame
    df = pd.DataFrame(compounds)
    
    # Display and save the results
    print(f"Retrieved {len(df)} chemical compounds from Wikidata")
    display(df.head(10))
    
    # Save to CSV
    df.to_csv('wikidata_chemical_compounds.csv', index=False)
    print("Data saved to 'wikidata_chemical_compounds.csv'")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

Querying Wikidata...
Retrieved 147275 chemical compounds from Wikidata


,0
0,Streptokinase
1,Scarlet GN
2,Rubidium hexafluorotitanate
3,Claziprotamide
4,Peregal O
5,Vicasinabin
6,PR-000608
7,Fluspidine
8,Aluminosilicate Refractory Ceramic Fibres
9,Lysergic acid pyrrolinide


Data saved to 'wikidata_chemical_compounds.csv'
